### Ingestão — Pipeline (pipelinevalor.globo.com)

Site inteiro (título + resumo), mesmo padrão de Estadão/O Globo/Valor-infra.
Testado isoladamente: funciona com httpx/curl_cffi, sem precisar de navegador.


In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()


In [0]:
# Carrega a função compartilhada que atualiza o status real da fonte na
# tabela de controle a cada execução.
%run "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/scripts/utils_controle"


In [0]:
import os
import re
import json
import time
import random
import hashlib
import unicodedata
import urllib.parse
from datetime import datetime, timezone
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests


In [0]:
HOJE = datetime.now(timezone.utc).astimezone().strftime("%Y-%m-%d")

SITE_URL = "https://pipelinevalor.globo.com/"
SOURCE_ID = "valor_pipeline"  # antes: hardcoded "site_page" no metadados; agora único

PASTA_DESTINO = f"/Volumes/desafio_kinea/research/research_volume/infraestrutura/sites/{HOJE}"
os.makedirs(PASTA_DESTINO, exist_ok=True)

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36",
]

IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]
HTTP_TIMEOUT = 30


In [0]:
def slugify(texto: str, max_len: int = 80) -> str:
    if not texto:
        return "sem-titulo"
    nfkd = unicodedata.normalize("NFKD", texto)
    ascii_txt = nfkd.encode("ascii", "ignore").decode("ascii")
    ascii_txt = re.sub(r"[^a-zA-Z0-9]+", "-", ascii_txt).strip("-").lower()
    return (ascii_txt[:max_len] or "sem-titulo").strip("-")


def hash_curto(texto: str, n: int = 8) -> str:
    return hashlib.md5(texto.encode("utf-8")).hexdigest()[:n]


def domain_from_url(url: str) -> str:
    try:
        netloc = urllib.parse.urlparse(url).netloc.lower()
        return netloc[4:] if netloc.startswith("www.") else netloc
    except Exception:
        return "desconhecido"


def headers_aleatorios(referer: Optional[str] = None) -> dict:
    headers = {"User-Agent": random.choice(USER_AGENTS), "Accept-Language": "pt-BR,pt;q=0.9"}
    if referer:
        headers["Referer"] = referer
    return headers


In [0]:
def baixar_html(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=url)
        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}] erro: {e}")

        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
        except Exception as e:
            print(f"  [httpx tent {tentativa}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None


def extrair_links(html: str, url_base: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    links, vistos = [], set()
    for tag_a in soup.find_all("a", href=True):
        href = tag_a["href"].strip()
        if not href or href.startswith(("#", "javascript:", "mailto:", "tel:")):
            continue
        url_absoluta = urllib.parse.urljoin(url_base, href)
        if url_absoluta in vistos:
            continue
        vistos.add(url_absoluta)
        links.append({"texto_ancora": tag_a.get_text(" ", strip=True), "url": url_absoluta})
    return links


In [0]:
TAGS_LIXO = ["script", "style", "noscript", "iframe", "svg", "form",
             "nav", "footer", "header", "aside", "button"]


def extrair_texto(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(TAGS_LIXO):
        tag.decompose()

    texto = soup.get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", texto).strip()


def extrair_titulo(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
        if soup.title and soup.title.string:
            return soup.title.string.strip()
    except Exception:
        pass
    return ""


In [0]:
def salvar_artefatos(pasta, source, titulo, texto, links, metadados):
    slug_source = slugify(source, max_len=40) or "fonte"
    slug_titulo = slugify(titulo, max_len=60) or "sem-titulo"
    sufixo_hash = hash_curto(metadados.get("url") or titulo)

    nome_base = f"{slug_source}_{slug_titulo}_{sufixo_hash}"
    caminho_txt = os.path.join(pasta, f"{nome_base}.txt")
    caminho_links = os.path.join(pasta, f"{nome_base}_links.json")
    caminho_json = os.path.join(pasta, f"{nome_base}.json")

    with open(caminho_txt, "w", encoding="utf-8") as f:
        f.write(texto or "")
    with open(caminho_links, "w", encoding="utf-8") as f:
        json.dump(links, f, ensure_ascii=False, indent=2)
    with open(caminho_json, "w", encoding="utf-8") as f:
        json.dump(metadados, f, ensure_ascii=False, indent=2)

    return caminho_txt, caminho_links, caminho_json


def processar_site(url: str, pasta_destino: str) -> Optional[dict]:
    print(f"=== Site: {url!r} ===")

    html = baixar_html(url)
    if not html:
        print("  -> download falhou; abortando.")
        return None
    print(f"  HTML baixado ({len(html)} chars).")

    links = extrair_links(html, url_base=url)
    texto = extrair_texto(html)
    print(f"  {len(links)} links, {len(texto)} chars de texto.")

    source = domain_from_url(url)
    titulo = extrair_titulo(html) or source
    metadados = {
        "source_id": SOURCE_ID,
        "title": titulo,
        "description": f"Página coletada diretamente do site: {url}",
        "url": url,
        "date": HOJE,
        "qtd_links": len(links),
        "qtd_chars_texto": len(texto),
    }

    caminho_txt, caminho_links, caminho_json = salvar_artefatos(
        pasta_destino, source, titulo, texto, links, metadados
    )
    print(f"  -> salvo em {caminho_txt}")

    return {"url": url, "titulo": titulo, "source": source, **metadados,
            "caminho_txt": caminho_txt, "caminho_links": caminho_links, "caminho_json": caminho_json}


In [0]:
try:
    resultado = processar_site(SITE_URL, PASTA_DESTINO)
except Exception as e:
    print(f"\n=== Fim. Erro inesperado: {e} ===")
    atualizar_status_fonte(source_id=SOURCE_ID, sucesso=False, docs_capturados=0, erro=str(e))
else:
    if resultado:
        print("\n=== Fim. Sucesso. ===")
        print(json.dumps(resultado, ensure_ascii=False, indent=2))
        atualizar_status_fonte(source_id=SOURCE_ID, sucesso=True, docs_capturados=1)
    else:
        print("\n=== Fim. Falhou. ===")
        atualizar_status_fonte(source_id=SOURCE_ID, sucesso=False, docs_capturados=0, erro="download ou extracao falhou")
